# K_𝖖-algebras — worked examples

This notebook is a guided, runnable tour of the public **`KAlgebra`** package:
a dependency-free (pure Python 3) implementation of the **K-theoretic Coulomb
branch algebras** `A_𝖖[T]` of 4d 𝒩=2 quantum field theories, following
Ambrosino–Gaiotto (DESY-25-035).

A **K_𝖖-algebra** is an associative algebra `A_𝖖` over `Z[𝖖^±]`, free as a
`Z[𝖖^±]`-module, equipped with

- a **bar involution** — an *antimultiplicative* involution sending `𝖖 ↦ 𝖖⁻¹`
  (so `A_𝖖 ≅ A_{𝖖⁻¹}^{op}`);
- a **canonical basis** `{L_a}` of bar-invariant elements, containing the unit;
- an algebra **automorphism `ρ`** permuting the canonical basis; and
- a **`ρ²`-twisted trace** `Tr`, under which the pairing
  $$I_{a,b} \;=\; \mathrm{Tr}\!\left(L_{\rho(a)}\,L_b\right) \;=\; \delta_{a,b} + O(\mathsf{q}).$$

That last relation — **orthonormality of the canonical basis to leading order
in `𝖖`** — is the defining constraint. `I_{a,b}(𝖖)` is the **Schur index** of
the pair of line defects `(a, b)`; orthonormality says the canonical basis is
orthonormal for the Schur pairing at `𝖖 → 0`, and it is rigid enough to
*determine* the trace up to normalization (from the single seed `Tr 1`).

The whole point of the codebase is that these axioms are made **executable**:
the abstract contract is the `KAlgebra` class (six primitives + derived axiom
*verifiers*), and each concrete theory is a subclass. Two presentations of the
same abstract algebra are certified equal by a `KAlgebraIso`.

The package is **layered**. This tour visits each layer in turn:

| step | layer | what it adds |
|---|---|---|
| 1 | `core` + `samples` | the contract, sample algebras, the quantum torus, `KAlgebraIso` |
| 2 | `cone` | exact, engine-free Schur indices for a large catalogue |
| 3 | `rg` | the **live RG-flow engine** — an algebra computed from its flow |
| 4 | `bps` | the **BPS-quiver engine** — an algebra discovered from its IR chart |
| 5 | `abe` | the **abelianized keystone** — pure U(N), constructive canonical basis |
| 6 | `skein` | **skein algebras of surfaces** realised as `A_𝖖[T[A₁, Σ]]` |

Every cell below runs in well under a second. All arithmetic is **exact**
`Z[𝖖^±]` / `R[𝖖^±]`; truncation to `O(𝖖^K)` happens only where a trace is read
out.

### How to run this notebook

The library is **pure Python 3 with no third-party dependencies**, so there is
very little to install. Launch Jupyter from the repository root (the directory
containing `src/`); the notebook locates `src/` automatically by walking up from
its own location. Pick whichever suits you:

- **In a browser, zero install (JupyterLite / Pyodide).** Because the code is
  pure Python, it runs unmodified in the browser via WebAssembly — no local
  Python, no `pip`. Open the notebook on a JupyterLite site and it just works.
- **JupyterLab / Jupyter Notebook (local, in your browser).**
  `pip install jupyterlab`, then `jupyter lab` in the repo folder — it opens a
  tab at `http://localhost:8888` and you run cells with **Shift+Enter**. Jupyter
  runs locally; nothing is uploaded.
- **VS Code.** Install the Python extension and `pip install ipykernel`, open
  this `.ipynb`, pick a Python interpreter, and click ▶ on cells.
- **No Jupyter at all — just check it runs.** From the repository root,
  `python3 notebooks/run_examples.py` executes every code cell headless and
  reports pass/fail. Needs nothing beyond Python 3.

In [1]:
# --- Path setup ---------------------------------------------------------
# The modules import one another by bare name; put every src/<layer>/ directory
# on sys.path (this is exactly what run_tests.py / conftest.py do).  The loop
# walks up from the current directory to find the repository's src/ tree.
import sys, pathlib

_here = pathlib.Path.cwd()
_src = None
for _base in (_here, *_here.parents):
    cand = _base / "src"
    if cand.is_dir() and (cand / "core").is_dir():
        _src = cand
        break
if _src is None:
    raise RuntimeError(
        "Could not locate the KAlgebra 'src/' tree. Run this notebook from the "
        "repository root (the directory containing README.md, run_tests.py, src/)."
    )

sys.path[:0] = [str(p) for p in _src.rglob("*")
                if p.is_dir() and p.name != "__pycache__"]

# Report which of the six layers are present, so a partial checkout is obvious.
_layers = ["core", "samples", "cone", "rg", "bps", "abe", "skein"]
_present = [d for d in _layers if (_src / d).is_dir()]
print("src/ tree:", _src)
print("layers present:", ", ".join(_present))

src/ tree: /tmp/claude-0/-home-user-Cluster/8f53a4b2-e505-563f-8c97-6ee5cae638b5/scratchpad/pub/KAlgebra/src
layers present: core, samples, cone, rg, bps, abe, skein

## Step 1 — The contract and the sample algebras (`core`, `samples`)

A concrete `KAlgebra` supplies **six primitives** — `coefficient_ring`,
`identity`, `multiply`, `rho`, `rho_inverse`, `trace` — and inherits the
derived bilinear extensions, the Schur pairing `inner_product`, and the axiom
**verifiers** (bar involution, `ρ`-automorphism, `ρ²`-twisted trace cyclicity,
orthonormality). The samples implement the six primitives *by hand*, so they
double as reference "how to write a K_𝖖-algebra".

### The quantum torus `Q_𝖖(Z²)` — the canonical example

For a lattice `Γ` with antisymmetric form `⟨·,·⟩`, the quantum torus has basis
`{L_γ}` with `L_γ · L_δ = 𝖖^{⟨γ,δ⟩} L_{γ+δ}`, bar fixing each `L_γ`, and
`ρ(γ) = −γ` (so `ρ² = id`). It is the auxiliary every BPS realisation reduces
to.

In [2]:
from samples import Z2QTorusSampleKAlgebra

A = Z2QTorusSampleKAlgebra()                 # Q_q(Z^2), the symplectic Z^2 torus
print("L_(1,0) · L_(0,1) =", A.multiply((1, 0), (0, 1)))   # q^<(1,0),(0,1)> L_(1,1)
print("L_(0,1) · L_(1,0) =", A.multiply((0, 1), (1, 0)))   # q^-1 L_(1,1)  (q-commuting)
print("rho(1,2)          =", A.rho((1, 2)))                # -gamma
print("axioms (rho aut.) =", A.verify_rho_is_automorphism((1, 0), (0, 1)))

# The Schur pairing I_{a,b} = Tr(rho(a)·b), an exact power series in q.
A.inner_product((1, 0), (1, 0), K=6)

L_(1,0) · L_(0,1) = (q)*L_(1, 1)
L_(0,1) · L_(1,0) = (q^-1)*L_(1, 1)
rho(1,2)          = (-1, -2)
axioms (rho aut.) = True

1 - 2*q^2 - q^4 + 2*q^6 + O(q^7)

The `inner_product` above carries the quantum-torus trace normalisation
`Tr L_γ = (𝖖²)_∞^{rk Γ} δ_{γ,0}`, i.e. the `(𝖖²)_∞²` Schur prefactor — its
`𝖖⁰` coefficient is `1`, an instance of orthonormality `I_{a,a} = 1 + O(𝖖)`.

### The pentagon `K_𝖖([A₁, A₂])` — Yang–Lee / the (2,5) minimal model

The simplest non-Lagrangian (Argyres–Douglas) example. Its five elementary
line defects are `L_i = (i, 1, 0)` for `i ∈ Z/5` (the label is `(i, a, b)`; the
unit is `(0, 0, 0)`). Multiplying two *non-adjacent* generators produces the
unit plus a lower generator — a characteristic fusion of the five-generator
(pentagon) structure. Orthonormality is visible directly in the Schur index.

In [3]:
from samples import PentagonSampleKAlgebra

P = PentagonSampleKAlgebra()
L0, L1, L2 = (0, 1, 0), (1, 1, 0), (2, 1, 0)     # elementary line defects L_0, L_1, L_2

print("L_0 · L_1 =", P.multiply(L0, L1))          # (q^-1)·L_(0,1,1)  — q-commuting
print("L_1 · L_0 =", P.multiply(L1, L0))          # (q)·L_(0,1,1)     — the opposite order
print("L_0 · L_2 =", P.multiply(L0, L2))          # 1 + (q^-1)·L_(1,1,0) — non-adjacent fusion

# The Schur index of an elementary line defect with itself: delta + O(q).
I = P.inner_product(L0, L0, K=6)
print("I_{L_0,L_0} =", I)                          # 1 - q^2 + q^4 + q^6 + ...
print("q^0 coefficient =", I[0], " (orthonormality: = 1)")

L_0 · L_1 = (q^-1)*L_(0, 1, 1)
L_1 · L_0 = (q)*L_(0, 1, 1)
L_0 · L_2 = L_(0, 0, 0) + (q^-1)*L_(1, 1, 0)
I_{L_0,L_0} = 1 - q^2 + q^4 + q^6 + O(q^7)
q^0 coefficient = 1  (orthonormality: = 1)

*Why orthonormality is an axiom, not an accident:* the Schur pairing counts
states with weight `𝖖^{Δ + spin}`; unitarity forces `Δ ≥ 0`, and only
`Spin(3)` highest weights contribute so `spin ≥ 0`. Hence the `𝖖⁰` term is
exactly the `Δ = spin = 0` identity sector, giving `I_{a,b} = δ_{a,b} + O(𝖖)`.

### A flavoured theory, and `base_change`

When the theory has a flavour symmetry `G_f`, the coefficient ring grows from
`Z` to the representation ring `R(G_f)` (a Lusztig–Ostrik **Z₊-ring**).
`SQED_{N_f}` = U(1) with `N_f` hypers has flavour `SU(N_f)`.

In [4]:
from samples import SQEDNfSampleKAlgebra
from zplus_ring import augmentation_hom

N = SQEDNfSampleKAlgebra(2)                   # SQED_2 = U_q(sl_2); flavour SU(2)
print("coefficient ring       :", N.coefficient_ring())        # SUNZPlusRing(2) = R(SU(2))

# base_change pushes the algebra along a ring hom; the augmentation chi_r -> dim r
# is the flavour-forgetful map, landing the algebra back over Z.
aug = augmentation_hom(N.coefficient_ring())
print("after augmentation     :", N.base_change(aug).coefficient_ring())   # Z

coefficient ring       : SUNZPlusRing(2)
after augmentation     : TrivialZPlusRing()

### `KAlgebraIso` — one algebra, many presentations

A `KAlgebraIso` certifies that two `KAlgebra` presentations are the *same*
abstract algebra (this is the object-identification goal). Here a
`B`-preserving frame change `M = [[1,1],[0,1]]` is an automorphism of the `Z²`
torus.

In [5]:
from kalgebra_iso import KAlgebraIso
from kalgebra import Element
from laurent_poly import LaurentPoly

A = Z2QTorusSampleKAlgebra()
ONE = LaurentPoly({0: 1})
M   = lambda g: (g[0] + g[1], g[1])          # unimodular frame change
Mi  = lambda g: (g[0] - g[1], g[1])
fwd = lambda l: Element({M(l): ONE})
inv = lambda l: Element({Mi(l): ONE})

iso = KAlgebraIso(A, A, fwd, inv, name="B-frame change")
pairs = [(Element({(1, 0): ONE}), Element({(0, 1): ONE}))]
img   = [(iso.map(x), iso.map(y)) for x, y in pairs]
print("iso is multiplicative :", iso.verify_multiplicative(pairs, img))

iso is multiplicative : True

## Step 2 — Cone realisations: exact Schur indices, no engine (`cone`)

A `ConeKAlgebra` presents the canonical basis as normal-ordered expressions in
a set of multiplicative **ray** generators. `multiply`, `trace` and
`verify_orthonormality` are **exact and arbitrarily `𝖖`-improvable**, computed
with *no realisation engine* (no quantum torus, RG, or BPS): every trace
reduces via `ρ²`-cyclicity to elementary seeds fixed by closed-form characters.

The pentagon again, now as `K_𝖖([A₁, A₂]) = A1A2kKAlg(1)`: its vacuum trace
`Tr(1)` is the character of the `(2,5)` (Yang–Lee) minimal model, and we read
it off exactly to order `𝖖⁴⁰`.

In [6]:
from a1a2k_kalg import A1A2kKAlg

A = A1A2kKAlg(1)                              # pentagon = K_q([A1, A2]), M(2,5)
print("Tr(1) to q^40 — the M(2,5) vacuum character:")
print(" ", A.trace(A.identity(), K=40))
print("orthonormality (vacuum with itself):", A.verify_orthonormality((), (), K=6))

Tr(1) to q^40 — the M(2,5) vacuum character:
  1 + q^4 + q^6 + q^8 + q^10 + 2*q^12 + 2*q^14 + 3*q^16 + 3*q^18 + 4*q^20 + 4*q^22 + 6*q^24 + 6*q^26 + 8*q^28 + 9*q^30 + 11*q^32 + 12*q^34 + 15*q^36 + 16*q^38 + 20*q^40 + O(q^41)
orthonormality (vacuum with itself): True

A larger, purely engine-free check: the finite-type `E₈` Argyres–Douglas
algebra has its canonical-basis orthonormality verified directly.

In [7]:
from finite_e8_kalg import FiniteE8KAlgebra

E8 = FiniteE8KAlgebra()
print("E8  coefficient ring       :", E8.coefficient_ring())
print("E8  orthonormality Tr rho(1)·1:", E8.verify_orthonormality((), (), K=6))

E8  coefficient ring       : TrivialZPlusRing()
E8  orthonormality Tr rho(1)·1: True

## Step 3 — The live RG-flow engine (`rg`)

An `RGKAlgebra` is a K_𝖖-algebra **defined by an RG flow** to a graded
auxiliary. The flow supplies only its data (`auxiliary`, `grading`, the
spectrum generator `S_RG`, `apex`); the *entire* contract API — `RG`,
`multiply`, `ρ`, `trace` — is then **computed live** from
`F_γ · S = X_γ + O(𝖖)` (the discovery relation), not tabulated.

### SQED₁ as a flow to the `Z²` quantum torus (the torus corner)

In [8]:
from u1_square_rg import U1SquareRGKAlgebra

A = U1SquareRGKAlgebra()                      # SQED_1 = U(1) + 1 hyper
print("multiply (1,0)·(0,1)      :", A.multiply((1, 0), (0, 1)))     # (q)·L_(1,1)
print("RG image of (0,1)         :", A.RG((0, 1)))                   # its quantum-torus image
print("trace (0,1) to q^6        :", A.trace((0, 1), 6))
print("F·S = X_gamma + O(q)?     :", A.verify_rg_discovery((0, 1)))  # the defining relation
print("orthonormality (0,1)      :", A.verify_orthonormality((0, 1), (0, 1), 6))

multiply (1,0)·(0,1)      : (q)*L_(1, 1)
RG image of (0,1)         : L_(0, 1)
trace (0,1) to q^6        : -q + q^5 + O(q^7)
F·S = X_gamma + O(q)?     : True
orthonormality (0,1)      : True

### The pentagon as a flow to SQED₁ (the non-torus corner)

Here the auxiliary is itself non-trivial. The engine *discovers* a canonical
basis element from a charge: `RG((2,1,0))` comes out supported on two
quantum-torus monomials, with bar-invariant coefficients.

In [9]:
from pentagon_square_rg import PentagonSquareSampleRGKAlgebra
from samples import PentagonSampleKAlgebra

A = PentagonSquareSampleRGKAlgebra()
Pdirect = PentagonSampleKAlgebra()
print("RG((2,1,0)) discovered as :", A.RG((2, 1, 0)))    # L_(0,-1) + L_(1,-1)

# The flow reproduces the direct pentagon's structure constants and trace.
a, b = (1, 1, 0), (0, 1, 0)
print("flow multiply   ==",  A.multiply(a, b))
print("direct multiply ==",  Pdirect.multiply(a, b))
print("traces agree (q^6):",
      A.trace((0, 0, 0), 6) == Pdirect.trace((0, 0, 0), 6))

RG((2,1,0)) discovered as : L_(0, -1) + L_(1, -1)
flow multiply   == (q)*L_(0, 1, 1)
direct multiply == (q)*L_(0, 1, 1)
traces agree (q^6): True

## Step 4 — The BPS-quiver realisation engine (`bps`)

A `BPSKAlgebra` realises the full contract from a **BPS quiver** — a Dirac
pairing `B` (antisymmetric) and node charges. It **discovers** the canonical
basis from the IR chart via the Kontsevich–Soibelman spectrum generator `S`
and the relation `F_γ · S = X_γ + O(𝖖)`, reads structure constants off the
auxiliary quantum torus, and computes the Schur-index trace exactly.

### The pentagon from its `A₂` quiver

In [10]:
from bps_kalgebra import BPSKAlgebra

B = BPSKAlgebra(pairing=[[0, 1], [-1, 0]], node_charges=[(1, 0), (0, 1)])
print("multiply (1,0)·(0,1)   :", B.multiply((1, 0), (0, 1)))       # (q)·L_(1,1)
print("Schur index I_{a,a}    :", B.inner_product((1, 0), (1, 0), K=6))
print("full axiom battery     :", B.verify_canonical_basis(K=6))

multiply (1,0)·(0,1)   : (q)*L_(1, 1)
Schur index I_{a,a}    : 1 - q^2 + q^4 + q^6 + O(q^7)
full axiom battery     : {'unital': True, 'multiplicative': True, 'bar_invariant': True, 'orthonormality': True}

`S` need not be supplied — with `build_S=True` it is built recursively from the
quiver alone (spec-free):

In [11]:
Bf = BPSKAlgebra(pairing=[[0, 1], [-1, 0]], node_charges=[(1, 0), (0, 1)],
                 build_S=True)
print("spec-free multiply     :", Bf.multiply((1, 0), (0, 1)))

spec-free multiply     : (q)*L_(1, 1)

### A flavoured theory: the hexagon

The hexagon quiver has `ker B = Z·(1,1,1)` — one flavour `U(1)`. The
coefficient ring becomes `R(U(1)) = Z[μ^±]`, and the flavour characters
`μ^{±k}` appear explicitly in the trace (shown as `[(±k,)]`).

In [12]:
H = BPSKAlgebra(pairing=[[0, 1, -1], [-1, 0, 1], [1, -1, 0]],
                node_charges=[(1, 0, 0), (0, 1, 0), (0, 0, 1)])
print("coefficient ring :", H.coefficient_ring())          # AbelianZPlusRing(rank=1)
print("Tr(vacuum) to q^4 (flavour characters [(±k,)] = mu^±k):")
print(" ", H.trace((0, 0, 0), K=4))

coefficient ring : AbelianZPlusRing(rank=1)
Tr(vacuum) to q^4 (flavour characters [(±k,)] = mu^±k):
  1 + ([(-1,)] + 1 + [(1,)])*q^2 + (2*[(-1,)] + [(-2,)] + 3 + 2*[(1,)] + [(2,)])*q^4 + O(q^5)

## Step 5 — The abelianized keystone: pure U(N) & Goal 2.1 (`abe`)

The third presentation tier: an algebra presented *faithfully on an enriched
rational quantum torus*. The flagship is `PureUNKAlgebra` — the pure U(N)
K-theoretic Coulomb branch algebra — labelled by **Kapustin `(m, λ)` charges**
(`m` a magnetic cocharacter, `λ` a Levi rep).

A defining discipline of this tier is the **constructive-build rule**: a
canonical-basis element is *built* as a polynomial in known bar-invariant
generators (dressed minuscules, `det`, Wilson lines), never *solved* for — a
solve could land outside the canonical span, where orthonormality would still
pass on the wrong element. Where a closed form does not reach, the tier
**honest-fails** rather than fabricate.

In [13]:
from pure_un_kalgebra import PureUNKAlgebra, default_rays

A = PureUNKAlgebra(2, default_rays(2), max_len=2, K=8)     # pure U(2)
Em  = ((1, 0), (0, 0))    # a minuscule 't Hooft monopole
Fm  = ((0, -1), (0, 0))   # its opposite
CHI = ((0, 0), (1, 0))    # the fundamental Wilson line
DET = ((1, 1), (0, 0))    # the determinant line

print("Em · Fm            :", A.multiply(Em, Fm))
# A Wilson line is a G-character; its fusion is the tensor product of reps:
# fund ⊗ fund = Sym^2 ⊕ det.
print("W_fund · W_fund    :", A.multiply(CHI, CHI))        # L_(2,0) + L_(1,1)
print("orthonormality (Goal 2.1):", A.verify_orthonormality(Em, Em, 4))
print("KL acceptance certify_canonical(Em) == Em:",
      A.certify_canonical(Em) == Em)

Em · Fm            : L_((1, -1), (0, 0))
W_fund · W_fund    : L_((0, 0), (1, 1)) + L_((0, 0), (2, 0))
orthonormality (Goal 2.1): True
KL acceptance certify_canonical(Em) == Em: True

Adding matter is *Abe + RG*: `UNNfKAlgebra(N, N_f)` presents U(N) with `N_f`
fundamental hypers, still with orthonormality holding on its canonical basis.

In [14]:
from un_nf_kalgebra import UNNfKAlgebra

M = UNNfKAlgebra(2, 1)                        # U(2) + N_f = 1
print("coefficient ring :", M.coefficient_ring())
print("orthonormality on Em, Em:", M.verify_orthonormality(((1, 0), (0, 0)),
                                                            ((1, 0), (0, 0)), 4))

coefficient ring : TrivialZPlusRing()
orthonormality on Em, Em: True

## Step 6 — Skein algebras of surfaces as `A_𝖖[T[A₁, Σ]]` (`skein`)

The SU(2) Kauffman-bracket **skein algebra** of a marked surface `Σ` is
realised as the K_𝖖-algebra of the class-S theory `T[A₁, Σ]`. Simple
non-self-intersecting curves are the ray generators; an ideal `n`-gon gives the
`A₁A_{n-3}` polygon family (the pentagon `n=5` is `A₁A₂`, the heptagon `n=7` is
`A₁A₄`), dispatched by `SkeinKAlgebra.polygon(n)`.

In [15]:
from skein_kalgebra import SkeinKAlgebra, ROSTER, PentagonSkeinKAlgebra

print("named instances in ROSTER :", len(ROSTER))
P5 = PentagonSkeinKAlgebra()                 # the pentagon, skein-side
print("pentagon-skein identity   :", P5.identity())

P7 = SkeinKAlgebra.polygon(7)                # A1A4 heptagon
print("polygon(7) sample product :", P7.multiply(((1, 0, 1),), ((1, 1, 1),)))

named instances in ROSTER : 17
pentagon-skein identity   : (0, 0)
polygon(7) sample product : (q^-1)*L_((2, 0, 1),) + L_()

The honest-fail discipline is first-class here too — asking for a polygon that
is out of range raises rather than guessing:

In [16]:
from skein_kalgebra import SkeinKAlgebra

try:
    SkeinKAlgebra.polygon(3)
except Exception as e:
    print("polygon(3) honest-fails:", type(e).__name__, "-", e)

polygon(3) honest-fails: NotImplementedError - SkeinKAlgebra.polygon(3): n must be >= 4

## Where to go next

- **Run the validation gate**: `python3 run_tests.py` — exercises the contract
  verifiers (bar involution, unit law, associativity, `ρ`-automorphism,
  `ρ²`-twisted trace cyclicity, orthonormality) across every layer.
- **Per-layer documentation**: `docs/step1-KAlgebra.md` … `docs/step6-SkeinKAlgebra.md`.
- **The conjectures**: `docs/conjectures-*.md` (the orthonormality conjecture
  and its verification scope), and `docs/axioms-and-bootstrap.md` (how the
  axioms determine the trace).

The two conjecture-families this codebase exists to test *and constructively
use*:

1. **Orthonormality of the canonical basis** — `I_{a,b}(𝖖) = δ_{a,b} + O(𝖖)`
   (seen in Steps 1, 2, 4, 5 above).
2. **RG intertwining** — an RG flow reproduces the algebra to leading order
   and intertwines the two `ρ`'s, `F·S = X_γ + O(𝖖)` being the leading case
   (Steps 3, 4).

All computation is exact in `Z[𝖖^±]`; the `O(𝖖)` is applied only at read-out.